시간날 때 다음 예제를 직접 해보기

https://ryuzyproject.tistory.com/104

# 1. 이안류 CCTV 데이터셋

[AI허브 이안류 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71297)

AI Hub의 '이안류 CCTV 데이터'는 우리나라 주요 해수욕장(해운대, 송정, 대천, 중문, 낙산)에서 이안류 발생 여부와 위치를 모니터링하기 위해 구축된 인공지능 학습용 데이터셋입니다.

해수욕장 주변에 설치된 CCTV 영상을 이미지로 변환하여, 이안류 발생 여부와 위치를 가시화하는 모델 개발에 활용할 수 있습니다.

이 데이터셋은 이안류 탐지 및 예측 시스템 개발에 필수적인 자료를 제공하며, 해수욕객의 안전을 위한 응용 서비스 구성에 활용될 수 있습니다.

이안류는 해안에서 먼 바다로 빠르게 이동하는 폭이 좁은 바닷물의 흐름으로, 기상 상태가 양호한 경우에도 나타나며, 얕은 곳에 있던 해수욕객을 순식간에 수심이 깊은 먼 바다로 이동시켜 인명사고를 유발할 수 있습니다.

따라서 이러한 데이터셋은 해수욕장 안전 관리 및 이안류 예측 모델 개발에 중요한 역할을 합니다.

# 2. YOLO 데이터셋 만들기
### 1. JSON Bounding Box > YOLO Bounding Box
- 원본 JSON의 drawing에는 객체를 둘러싼 사각형의 점 좌표가 들어있음
- YOLO의 일반적인 객체 탐지 라벨 형식
    - `클래스id`, `x센터`, `y센터`, `너비`, `높이`
    - 네개의 좌표를 모두 0 ~ 1 사이로 정규화

### 2. 원본 이미지와 JSON 라벨을 연결한 뒤 train/val/test 폴더로 나눔
- 같은 원본 영상에서 나온 프레임이 train과 test에 동시에 들어가면 성능이 실제보다 높게 보일 수 있음
- 파일명 앞부분을 그룹으로 사용해 가능한 한 같은 영상 계열이 한 ㅅ트에만 들어가도록 분리
- 데이터 수가 적으면 YOLO 성능이 낮아질 수 있음 (전체 데이터셋을 가지고 써보면 좋겠다는 의미)
    - 샘플 데이터에는 전체 이미지가 360개밖에 없어서
        - 전체 이미지: 360
        - 이안류 객체가 있는 이미지: 240장
        - 배경이미지: 120장
- YOLO는 단순히 이미지를 외우는 것이 아니라, 여러 이미지에서 반복적으로 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야함.

> YOLO는 단순히 이미지를 외우는 것이 아니라, 여러 이미지에서 반복적으로 나타나는 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야 함. 단숞한 파일 개수가 아니라 서로 다른 상황을 얼마나 다양하게 포함하고 있는지가 중요

- CCTV 연속 프레임에서는 무작위로 이미지를 섞어 나누는 방법을 사용하지 않음
    - 서로 몇초 차이인 것의 동일한 프레임이 Train과 Test를 동시에 들어가면 모델은 Test 이미지를 처음 보는 것이 아니라, 학습 때 본 장면과 매우 유사한 장면을 다시 보는 것과 비슷해짐 (데이터 누수)
- 전체 데이터셋을 확보할 수 있다면 아래 단위로 분리하는 것이 좋음
    - CCTV 카메라 각도
    - 해수욕장 위치
    - 날짜
    - 촬영 시간대


# **3. Detection이 문제 정의에 적합한지 검토**
> 이안류 영역을 Bounding Box로 감싸는 ObjectDetection 방식은 이안류처럼 형태가 불규칙하고 경계가 명확하지 않은 영역일 경우 Segmentation이 더 적합할 수 있음

# 여기서부터 직접 해보는 실습
-----

------


## 1. 우선 작업을 하기 전에 고려해야할 사항
1. 일단 모델을 가져와야하며 (YOLO) 이것을 미리 사전학습을 시킬 수 있으면 좋을 것 같습니다.
2. 그리고 이안류 데이터셋을 가져와서 어떤식으로 데이터가 만들어져있는지, 카메라의 위치가 나뉘어져 있는지, 시계열성 데이터가 있는 것인지, 또는 데이터에 어떤 정보가 있는지를 확인해야할 것 같습니다. 이에 따라 group 기준 분할을 해야할 수도 있을 것 같습니다. 추가적으로 이안류/비이안류 비율도 층화추출을 통해 비율을 맞추어야할 것 같습니다.
3. 데이터셋이 작지 않기 때문에 DataLoader을 API 호출 방식으로 사용하여야할지도 고려 대상인 것 같습니다.
4. 이안류 데이터셋을 판단할 때 이를 판단할 평가 지표를 미리 설계하고 시작해야할까 싶으며 동시에 여러 실행 환경 (옵티마이저, 손실함수, 평가지표, 데이터 분할, 스케줄링 등을 계획해야겠다고 생각합니다.)

## 2. 이안류란 무엇인가?
이안류 `rip current`는 파도 해안에서 바다쪽으로 빠져나가는 상대적으로 좁고 강한 흐름을 이야기합니다.

여기에서 이안류 이미지는 바운딩 박스로 표현되어있기 때문에 객체탐지를 통해 대략적인 이안류의 위치를 알 수 있게 될 것 같습니다.

아래 이미지는 샘플 데이터에 존재하는 이미지의 예시이며 

이미지 파일에는 아마 찍은 장치의 화질등에 대한 값인 `DC_ZIPTR`, `HD_GLORY` 등과 찍은 `YYYYMMDD_HHMMSS` 형태가 함께 붙어있습니다.

![sample_image](./DC_MUDCH_20190705_133730.jpg)

![image.png](./222518.png)

또한 라벨데이터의 경우에는 내부적으로 아래와 같이 상태가 존재하였습니다. 

주요 데이터로는 아마 `image_info`의 `GPS`, 바운딩 박스를 의미하는 `drawing` 등이 있을 것 같으며 `class`의 경우에는 지금 `1`이 존재하거나 아예 존재하지 않거나 뿐이며 `bounding_count` 또한 1 이상인 데이터가 보이지 않는 것 같습니다.

```json
{
    "image_info": {
        "file_name": "SJ_WHIB1_20190617_091236.jpg",
        "collection_method": "Null",
        "ID_code": "SJ_WHIB1",
        "date": "Null",
        "GPS": "35.1767,129.1986",
        "resolution": "1920,1080",
        "focus_distance": "Null",
        "model_name": "Null",
        "image_size": "Null"
    },
    "annotations": {
        "bounding_count": 1,
        "class": 1,
        "details": "Clear",
        "sun_light": "Null",
        "object": "Yes",
        "drawing": [
            [
                [ 492, 552 ],
                [ 793, 552 ],
                [ 793, 656 ],
                [ 492, 656 ]
            ]
        ],
        "rip_current_duration": null,
        "rip_current_phase": null,
        "significant_wave_height": 1.71,
        "significant_wave_period": 8.77,
        "wind_velocity": 4.9,
        "wind_direction": 196.0,
        "wave_direction_sprading_factor": 29.7939,
        "spectrum_spreading_factor": 1.9956,
        "peak_period": 10.24,
        "peak_direction": 59.0,
        "angle_of_incidence_of_the_beach": 142,
        "tide_level": 1.21
    }
}
```

> 위 데이터중에서 실제로 사용할 부분은 class, drawing(bbox), bounding_count 정도이지 않을까 싶으며, 데이터를 나눌 때에는 GPS 정도 쓰일까 싶습니다.

In [7]:
# 데이터 폴더 지정
from pathlib import Path

DATA_DIR = Path() / "data" / "rip-current"
LABELING_DIR = DATA_DIR / "02.라벨링데이터"
SOURCE_DIR = DATA_DIR / "01.원천데이터"

print(DATA_DIR.resolve())
print(DATA_DIR.exists())

C:\HUR\Document\lang-chain-KDT\01_class\computer_vision\18_computer_vision_opencv_lessons\data\rip-current
True


In [81]:
from datetime import datetime

# 라벨링 데이터를 뜯어서 class의 종류가 어떤 것이 있는지 확인하기
import json

# 클래스 종류 뽑아보기 / bounding_count 뽑아보기
classes = set()
bounding_count = set()

# gps lat/lon 뽑아보기
lat = set()
lon = set()

# 찍은 시간 분포 뽑기 위한 준비
time_distribute = dict()

ct_gps = dict()

file_count = 0

bbox_err_flag = False

camera_type = set()

id_code_gps_not_match = False

# 이안류 유무 데이터 개수
rip_bool = {
    0: 0,
    1: 0
}

# 모든 라벨링 데이터 순회하기
for file in LABELING_DIR.iterdir():
    # 혹시 모르니 파일 검사
    if file.is_file() and file.name.endswith(".json"): 
        file_count += 1
        with open(file, "r", encoding="utf-8") as f:

            

            obj = json.load(f)
            # print(type(obj["image_info"].get("GPS")))

            # 카메라의 종류
            # ct = file.name.split("_")[0]
            # camera_type.add(ct)
            ct = obj["image_info"].get("ID_code")

            # 한 카메라가 하나의 GPS에 속하는지 확인
            if ct_gps.get("ct") is None:
                ct_gps[ct] = obj["image_info"].get("GPS")
            else:
                if ct_gps[ct] != obj["image_info"].get("GPS"):
                    id_code_gps_not_match = True

            # 이안류 유무 누적
            if rip_bool.get(obj["annotations"].get("object")) is None:
                rip_bool[obj["annotations"].get("object")] = 1
            else:
                rip_bool[obj["annotations"].get("object")] += 1

            # 카메라별 촬영 시간 뽑아주기
            yyyymmdd, hhmmss = file.name.split(".")[-2].split("_")[-2:]

            yyyymmddhhmmss = int(f"{int(yyyymmdd):08d}{int(hhmmss):06d}")

            unix_time = datetime.strptime((yyyymmdd + hhmmss), "%Y%m%d%H%M%S")

            if time_distribute.get(ct) is None:
                time_distribute[ct] = set([unix_time])
            else:
                time_distribute[ct].add(unix_time)
            
            # 위도, 경도 추가해주기
            lat_add, lon_add = obj["image_info"].get("GPS").split(",")
            lat.add(lat_add)
            lon.add(lon_add)

            # 클래스 종류와 바운딩 박스 뽑아보기
            classes.add(obj["annotations"].get("class"))

            # 클래스가 있다면 
            if obj["annotations"].get("drawing") is not None:
                for data in obj["annotations"].get("drawing"):
                    # xmin, xmax, ymin, ymax 형태가 맞는지 확인
                    data_sum = []
                    for d in data:
                        data_sum += d

                    if not (
                        data_sum[0] == data_sum[6] 
                        and data_sum[1] == data_sum[3] 
                        and data_sum[2] == data_sum[4] 
                        and data_sum[5] == data_sum[7]):
                        bbox_err_flag = True

            
            
            # 이안류가 3개 이상인 것: 샘플 데이터에서는 3개
            # if obj["annotations"].get("bounding_count") >= 3:
            #     print(file.name)
            bounding_count.add(obj["annotations"].get("bounding_count"))

print("file_count:", file_count)

print("classes:", classes)
print("bounding_count:", bounding_count)

# time_cluster = 0
# 시간들 중에서 5초 간격 내에 있는 것은 하나의 집합으로 판단해주기
# for t1, t2 in zip(time_distribute[:-1], time_distribute[1:]):
#     if (t2 - t1).total_seconds() > 10000:
#         time_cluster += 1

print()
print("카메라별 2초 보다 떨어진 간격들")
for name, times in time_distribute.items():
    time_cluster = 0
    times = sorted(list(times))
    for t1, t2 in zip(times[:-1], times[1:]):
        if (t2 - t1).total_seconds() > 2:
            time_cluster += 1
    print("camera:", name, "| time_cluster:", time_cluster)

# 3, 5, 10, 15 등 시도해봤는데 전체적인 느낌은 
# print(time_cluster)

print()
print("is not bbox:", bbox_err_flag)

print(len(lat), len(lon))

print("camera_type count:", len(camera_type))

print("아이디의 GPS가 고정인지:", not id_code_gps_not_match)            

file_count: 360
classes: {0, 1}
bounding_count: {0, 1, 2, 3}

카메라별 2초 보다 떨어진 간격들
camera: DC_MUDCH | time_cluster: 2
camera: DC_ZIPTR | time_cluster: 2
camera: HD_GLORY | time_cluster: 7
camera: HD_PARA1 | time_cluster: 6
camera: HD_PARA2 | time_cluster: 5
camera: HD_SEAC1 | time_cluster: 5
camera: JM_BADA1 | time_cluster: 1
camera: JM_BADA2 | time_cluster: 1
camera: NS_NSBE1 | time_cluster: 2
camera: NS_NSBE2 | time_cluster: 13
camera: SJ_SJHT1 | time_cluster: 9
camera: SJ_WHIB1 | time_cluster: 7

is not bbox: False
8 8
camera_type count: 0
아이디의 GPS가 고정인지: True


위의 결과를 통해 lat/lon의 개수가 한정되어있으며 (8개의 카메라 위치) 클래스는 0 또는 1 (이안류가 있거나 없거나)이라는 것을 알 수 있었습니다.

또한 `bounding_count`가 0에서 3까지 있어서 이안류가 복수 있을 수 있다는 것을 확인할 수 있었습니다.

위에서 구한 시간의 경우는 모든 카메라를 기준으로 판단한 것이기 때문에 더 올바른 방식이 있지만 어느정도 시간대는 군집화되어있다고 알 수 있습니다.

# YOLO가 기대하는 파일 구조와 라벨 형식 잡기
현재는 데이터에 `01.원천데이터`인 `.jpg` 데이터가 있으며 `02.라벨링데이터`인 `.json` 데이터가 있습니다. 또한 라벨링 데이터에는 `xmin`, `xmax`, `ymin`, `ymax` 형태로 이루어져 있습니다.

여기에서 YOLO Detection 형태로 변환하기 위해서는 

루트에 `data.yaml` 형태가 있으며 `train/`, `val/`, `test/` 가 존재하는 형태가 되어야합니다.

최종적으로는
- `data.yaml`
    - `iamges/`
        - `train/`
        - `val/`
        - `test/`
    - `labels/`
        - `train/`
        - `val/`
        - `test/`

형태로 이루어져있어야 합니다.

여기에서 위경도 분류는 총 8개 (샘플데이터 기준)이며 하나의 카메라 id에는 gps가 고정이기 때문에 id를 기준으로 이미지들을 식별한 뒤, id별로 train/val/test 를 나누는 것도 괜찮아 보입니다

여기에서 고민한 것이 시간대별로 나누는 것 또는 공간별로 나누는 것에서 생각을 해보았는데 일단 **새로운 위치에서 식별을 잘 해내는가**에 대해서 알아보는 것이 나쁘지 않을 것 같아서 `id`별로 나누는 것으로 선택하였습니다.

> 이렇게 선택한 이유에는 미래 예측보다는 우선 객체탐지가 우선이기 때문이였으며, 여유가 될 경우 여러가지 방식을 비교해가는 것도 좋을 것 같습니다.

# 데이터 받는중...
현재 API 방식으로 DataLoader을 구현하는 것을 생각하였지만 결국 데이터의 정확한 분할을 위해서 모두 받아오는 것이 맞다고 판단하여 32GB 정도를 통채로 받기로 했습니다.

In [86]:
# 미리 json을 yolo형식으로 바꾸는 함수 만들어놓기
def rip_json_to_yolo_annot(json_file_path: Path, target_dir: Path) -> None:
    """json 파일 Path 객체를 받아서 target_dir에 형식만 바꾸어서 yolo 형식의 .txt로 바꾸어줍니다."""

    # 파일을 열어주기
    with open(json_file_path, "r", encoding="utf-8") as f:
        # 객체 읽어오기
        obj = json.load(f)

        # 작성할 텍스트 준비
        input = ""

        # 객체에서 위치를 변형해주기. class가 있으면 항상 1임
        if obj["annotations"].get("class") == 1:
            # 이미지 전체 크기 잡아주기
            im_w, im_y = obj["image_info"].get("resolution", "").split(",")
            im_w, im_y = int(im_w), int(im_y)
            for bbox in obj["annotations"].get("drawing"):
                # 배열 분해해서 받아주기
                p1, _, p3, _ = bbox

                # 다시 한번 더 분해해주기
                xmin, ymin = p1
                xmax, ymax = p3

                # 그대로 계산과 함께 넣어주기
                x_cen = (xmin+xmax) / 2
                y_cen = (ymin+ymax) / 2

                # w, h 구해주기
                w = xmax-xmin
                h = ymax-ymin

                input += f"1 {x_cen/im_w} {y_cen/im_y} {w} {h}\n"

        with open((target_dir / ".".join(file.name.split(".")[:-1] + ["txt"])), "w", encoding="utf-8") as fw:
            fw.write(input)

In [87]:
# 잘 동작하는지 확인
rip_json_to_yolo_annot(
    Path(r"C:\HUR\Document\lang-chain-KDT\01_class\computer_vision\18_computer_vision_opencv_lessons\data\rip-current\02.라벨링데이터\SJ_SJHT1_20190607_091019.json"),
    Path(r"C:\HUR\Document\lang-chain-KDT\01_class\computer_vision\18_computer_vision_opencv_lessons\data\rip-current")
)

# 함수는 잘 동작합니다.
![image-3.png](./img2.png)